## Install Libraries

In [1]:
!pip install langchain langchain-openai openai wikipedia langchain-community langchain-experimental

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=080d377001914fc121b246f329498abd2d0dafd0ed71579163eef628492f9428
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
  Attempting uninstall: requests
    F

In [2]:
import os
from google.colab import userdata
OpenAI_API = userdata.get('OPENAI_API_KEY')  # Fetching the stored OpenAI API key from the Colab userdata storage
os.environ['OPENAI_API_KEY'] = OpenAI_API  # Setting the fetched API key as an environment variable for use in the application

## React Agent with tools

A ReAct agent combines reasoning and acting to solve a user’s request. Instead of generating an answer immediately, the agent determines what information or calculation is required, selects an appropriate tool, observes the tool’s result, and then produces the final response.

The ReAct workflow generally follows this pattern:

Understand the user’s request.
Decide whether an external tool is required.
Select and call the appropriate tool.
Observe the result returned by the tool.
Continue reasoning or call another tool if necessary.
Generate the final answer.

For example, an agent may use:

A calculator tool for mathematical operations
A Wikipedia tool for general knowledge
A search tool for retrieving current information
A database tool for accessing structured data
An API tool for performing external actions

In earlier LangChain versions, this approach was commonly implemented using a Zero-Shot ReAct Description Agent. The term zero-shot means that the agent can select and use tools based only on their names and descriptions, without requiring task-specific training examples.

In current LangChain versions, agents are created using the create_agent() function. The agent receives a language model, a collection of tools, and a system prompt describing its expected behavior.

In [4]:
import ast
import operator
import os

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.checkpoint.memory import InMemorySaver

/tmp/ipykernel_551/1947767720.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [5]:
# Initialize the OpenAI chat model.
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
)

In [9]:
# Configure the Wikipedia API wrapper.
wikipedia_api = WikipediaAPIWrapper(
    top_k_results=3,
    doc_content_chars_max=4000,
)


# Create the Wikipedia search tool.
wikipedia_tool = WikipediaQueryRun(
    api_wrapper=wikipedia_api,
)


# Operators permitted inside the calculator tool.
ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def evaluate_expression(node):
    """Recursively evaluate a safe arithmetic expression."""

    # Handle numeric constants such as 5 or 3.5.
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    # Handle older Python numeric AST nodes.
    if isinstance(node, ast.Constant):
        return node.n

    # Handle binary operations such as addition and multiplication.
    if isinstance(node, ast.BinOp):
        operator_function = ALLOWED_OPERATORS.get(type(node.op))

        if operator_function is None:
            raise ValueError("Unsupported arithmetic operator.")

        left_value = evaluate_expression(node.left)
        right_value = evaluate_expression(node.right)

        return operator_function(left_value, right_value)

    # Handle unary operations such as negative numbers.
    if isinstance(node, ast.UnaryOp):
        operator_function = ALLOWED_OPERATORS.get(type(node.op))

        if operator_function is None:
            raise ValueError("Unsupported unary operator.")

        return operator_function(evaluate_expression(node.operand))

    raise ValueError("The expression contains unsupported syntax.")


@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.

    Use this tool for arithmetic calculations. The input must be a valid
    arithmetic expression, such as '(3 * 2) + (2 * 3)'.
    """

    try:
        parsed_expression = ast.parse(
            expression,
            mode="eval",
        )

        result = evaluate_expression(parsed_expression.body)

        return str(result)

    except ZeroDivisionError:
        return "Error: division by zero is not allowed."

    except (SyntaxError, TypeError, ValueError) as error:
        return f"Error: invalid mathematical expression. {error}"


# Collect the tools that can be called by the agent.
tools = [
    wikipedia_tool,
    calculator,
]

In [10]:
# Create the general-purpose tool-calling agent.
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful research and calculation assistant.

    Use the calculator tool for arithmetic calculations.
    Use the Wikipedia tool for factual and encyclopedic questions.

    Carefully interpret the user's question before choosing a tool.
    Explain the final result clearly and concisely.
    """,
)


# Invoke the agent for the arithmetic question.
result_1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
                I am going to buy 5 boxes of apples. Each box contains
                2 apples. If I do not buy 5 boxes of apples and instead
                buy 3 boxes of apples and 2 boxes of oranges, where each
                orange box contains 3 oranges, how many total apples
                will I have?
                """,
            }
        ]
    }
)


# Extract and print the final response.
print(result_1["messages"][-1].content)

If you buy 5 boxes of apples, with each box containing 2 apples, you will have a total of 10 apples.

If instead you buy 3 boxes of apples (which gives you 6 apples) and 2 boxes of oranges (which gives you 6 oranges), you will still have 6 apples from the apple boxes.

Therefore, if you choose to buy 3 boxes of apples and 2 boxes of oranges, you will have a total of **6 apples**.


In [11]:
# Invoke the agent for the Wikipedia question.
result_2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
                What is the highest mountain peak in the world?
                What are the next two highest mountain peaks?
                """,
            }
        ]
    }
)


# Extract and print the final response.
print(result_2["messages"][-1].content)

The highest mountain peak in the world is **Mount Everest**, which stands at an elevation of **8,848.86 meters** (29,031.7 feet) above sea level.

The next two highest mountain peaks are:
1. **K2** (Mount Godwin-Austen) at **8,611 meters** (28,251 feet).
2. **Kangchenjunga** at **8,586 meters** (28,169 feet).

These peaks are part of the Himalayas and the Karakoram mountain ranges, primarily located in Nepal, India, and Pakistan.


## Add memory to Agent

In [12]:
from langgraph.checkpoint.memory import InMemorySaver

In [13]:
# Create an in-memory checkpointer.
# This stores the conversation state for each thread.
memory = InMemorySaver()

In [14]:
# Create an agent with short-term conversational memory.
conversational_agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,
    system_prompt="""
    You are a helpful conversational assistant.

    Remember information from earlier messages in the same conversation.
    Use Wikipedia for encyclopedic questions.
    Use the calculator for arithmetic calculations.
    Give clear and concise answers.
    """,
)


# All calls using this thread ID share the same conversation history.
config = {
    "configurable": {
        "thread_id": "user-conversation-1"
    }
}

In [15]:
# First conversational turn.
result_3 = conversational_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who won the 2022 FIFA World Cup?",
            }
        ]
    },
    config=config,
)


print(result_3["messages"][-1].content)

Argentina won the 2022 FIFA World Cup, defeating France in the final. The match ended in a 3–3 draw after extra time, and Argentina triumphed 4–2 in the penalty shootout. This victory marked Argentina's third World Cup title, their first since 1986.


In [16]:
# Second turn using the same thread ID.
# The agent can remember that the previous topic was the 2022 World Cup.
result_4 = conversational_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Which team did they defeat in the final?",
            }
        ]
    },
    config=config,
)


print(result_4["messages"][-1].content)


# Third turn using the same conversation.
result_5 = conversational_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What was the final score?",
            }
        ]
    },
    config=config,
)


print(result_5["messages"][-1].content)

Argentina defeated France in the final of the 2022 FIFA World Cup.
The final score of the 2022 FIFA World Cup final was 3–3 after extra time, and Argentina won 4–2 in the penalty shootout.
